In [1]:
import pandas as pd
import numpy as np
from scipy.stats import qmc
from sklearn.preprocessing import StandardScaler as SS
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
import xgboost as xgb
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
from sklearn.exceptions import ConvergenceWarning
import plotly.express as px
import itertools
from sklearn.base import clone

In [2]:
#GENERAZIONE DATASET X SIMULAZIONI
lista_zavorra = [0, 20, 40, 60]
campioni_per_zavorra = 50

risultati = []

#LHS
sampler = qmc.LatinHypercube(d=4, seed=999)

for zavorra in lista_zavorra:
    #RANGE DELAY IN BASE ALLA ZAVORRA
    if zavorra == 0:
        lim_delay = [1.0, 3.0]
    elif zavorra == 20:
        lim_delay = [0.9, 2.9]
    elif zavorra == 40:
        lim_delay = [0.8, 2.8]
    elif zavorra == 60:
        lim_delay = [0.8, 2.7]
    
    limiti_inferiori = [0.0, -5.0, 0.0, lim_delay[0]]
    limiti_superiori = [6.0, -3.0, 0.25, lim_delay[1]]
    
    campioni_lhs = sampler.random(n=campioni_per_zavorra)
    campioni_scalati = qmc.scale(campioni_lhs, limiti_inferiori, limiti_superiori)

    for riga in campioni_scalati:
        vento = round(riga[0], 2)
        coeff_teta = round(riga[1], 2)
        std_vento = round(riga[2], 3)
        delay = round(riga[3], 2)

        teta_rampa = round(vento * coeff_teta, 2)
        
        if vento < 0.2:
            teta_rampa = 0.0

        risultati.append({
            'vento_medio': vento,
            'std_vento': round(std_vento * vento, 2),
            'zavorra': zavorra,
            'delay_secondi': delay,
            'angolo_rampa': teta_rampa,
            'APOGEO_MISURATO': '',
            'ANGOLO_ACC_C6': '',
            'INCLINAZIONE': ''
        })

df = pd.DataFrame(risultati)
df.to_csv("Simulazioni_GEMINI-v2.csv", index=False)

In [2]:
df = pd.read_csv("Simulazioni_GEMINI-v3.csv")

#"DATA CLEANING"
for column in df.columns:
    df[column] = df[column].astype("str")
    df[column] = df[column].str.replace(",", ".")
    df[column] = df[column].str.replace(r'[^0-9.\-]', '', regex=True) 
    df[column] = df[column].apply(lambda x: str(x).split('.')[0] + '.' + str(x).split('.')[1] if len(str(x).split('.')) > 2 else x)
    df[column] = pd.to_numeric(df[column], errors='coerce')

df = df.dropna()

#APPLICA SEGNO INCLINAZIONE
moltiplicatori_perfetti = {
    0.0: -4.2,
    20.0: -4.0,
    40.0: -3.8,
    60.0: -3.5}

def applica_segno_inclinazione(riga):
    zavorra = float(riga["zavorra"])
    vento = float(riga["vento_medio"])
    angolo_rampa = float(riga["angolo_rampa"])
    angolo_misurato = float(riga["ANGOLO_ACC_C6"])
    
    err_assoluto = 90.0 - angolo_misurato
    
    if vento == 0.0 or err_assoluto == 0.0:
        return err_assoluto
        
    molt_perfetto = moltiplicatori_perfetti.get(zavorra, -4.0)
    angolo_perfetto = vento * molt_perfetto
    
    if angolo_rampa < angolo_perfetto:
        return -abs(err_assoluto)
    else:
        return abs(err_assoluto)

df["INCLINAZIONE_ANGOLO"] = df.apply(applica_segno_inclinazione, axis=1)

#SHUFFLE
df = df.sample(frac=1, random_state=999).reset_index(drop=True)

#TARGET VARIABLES
y1 = 0.9 * df["APOGEO_MISURATO"]
y2 = df["INCLINAZIONE_ANGOLO"]

#PREDICTIVE VARIABLES
x = df.drop(["APOGEO_MISURATO", "ANGOLO_ACC_C6", "INCLINAZIONE_ANGOLO"], axis=1)

#NORMALIZZAZIONE
scaler = SS()
x_norm = scaler.fit_transform(x)

In [3]:
#RANDOM FOREST
modello_rf = RandomForestRegressor(
    n_estimators=100,
    random_state=999,
    n_jobs=-1)

#XGBOOST
modello_xgb = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=999,
    n_jobs=-1)

#SUPPORT VECTOR REGRESSION
modello_svr = SVR(
    kernel='rbf',
    C=50, #100,
    gamma='scale')

#NN
nn = MLPRegressor(
  hidden_layer_sizes=(16, 8),
  activation='tanh',
  solver='lbfgs',
  alpha=0.1,
  max_iter=5000,
  tol=0.001,
  random_state=999)

modelli_livello_0 = [('m1', modello_rf), ('m2', modello_xgb), ('m3', modello_svr), ('m4', nn)]

In [4]:
#STACKING
stacking = StackingRegressor(
  estimators=modelli_livello_0,
  final_estimator=Ridge(),
  cv=5,
  n_jobs=-1)

In [5]:
#FUNZIONE PER PREVISIONI
def val(modello, dati_x, reale):
    kf = KFold(n_splits=5, shuffle=True, random_state=999)
    return cross_val_predict(modello, dati_x, reale, cv=kf)

#METRICHE MODELLI
def valuta_metriche(target, reale, previsioni, maschera):
    reale_filtrato = reale[maschera]
    previsioni_filtrate = previsioni[maschera]
    
    mae = mean_absolute_error(reale_filtrato, previsioni_filtrate)
    rmse = np.sqrt(mean_squared_error(reale_filtrato, previsioni_filtrate))
    bias = np.mean(previsioni_filtrate - reale_filtrato)
    
    print(f"Risultati modello x previsione {target} (su {len(reale_filtrato)} lanci approvati)")
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"Bias: {bias:.2f}\n")

In [6]:
print("Cucinando...\n")

#PREVISIONI
prev_apo_stack = val(stacking, x_norm, y1)
prev_ang_stack = val(stacking, x_norm, y2)

prev_apo_xgb = val(modello_xgb, x_norm, y1)
prev_ang_xgb = val(modello_xgb, x_norm, y2)

prev_apo_rf = val(modello_rf, x_norm, y1)
prev_ang_rf = val(modello_rf, x_norm, y2)

#MASCHERA
tolleranza = 10
maschera_stack = np.abs(prev_ang_stack) <= tolleranza
maschera_xgb = np.abs(prev_ang_xgb) <= tolleranza
maschera_rf = np.abs(prev_ang_rf) <= tolleranza

#VALUTAZIONE
print("Modello: STACKING")
valuta_metriche("APOGEO", y1, prev_apo_stack, maschera_stack)
valuta_metriche("ERRORE INCLINAZIONE", y2, prev_ang_stack, maschera_stack)

print("Modello: XGBOOST")
valuta_metriche("APOGEO", y1, prev_apo_xgb, maschera_xgb)
valuta_metriche("ERRORE INCLINAZIONE", y2, prev_ang_xgb, maschera_xgb)

print("Modello: RANDOM FOREST")
valuta_metriche("APOGEO", y1, prev_apo_rf, maschera_rf)
valuta_metriche("ERRORE INCLINAZIONE", y2, prev_ang_rf, maschera_rf)

Cucinando...

Modello: STACKING
Risultati modello x previsione APOGEO (su 135 lanci approvati)
MAE: 6.79
RMSE: 10.25
Bias: -3.36

Risultati modello x previsione ERRORE INCLINAZIONE (su 135 lanci approvati)
MAE: 7.09
RMSE: 10.62
Bias: -1.41

Modello: XGBOOST
Risultati modello x previsione APOGEO (su 122 lanci approvati)
MAE: 7.40
RMSE: 11.58
Bias: -2.59

Risultati modello x previsione ERRORE INCLINAZIONE (su 122 lanci approvati)
MAE: 7.09
RMSE: 10.58
Bias: -1.17

Modello: RANDOM FOREST
Risultati modello x previsione APOGEO (su 136 lanci approvati)
MAE: 7.29
RMSE: 10.79
Bias: -2.74

Risultati modello x previsione ERRORE INCLINAZIONE (su 136 lanci approvati)
MAE: 7.95
RMSE: 11.92
Bias: 0.24



In [7]:
#ANALISI RESIDUI INCLINAZIONE - SCATTER PLOT
prev_ang = prev_ang_stack 

tolleranza = 5
maschera = np.abs(prev_ang) <= tolleranza

reale_ang_filtrato = y2[maschera]
prev_ang_filtrato = prev_ang[maschera]

residui_ang = reale_ang_filtrato - prev_ang_filtrato

df_residui = pd.DataFrame({
    'Reale': reale_ang_filtrato,
    'Previsto': prev_ang_filtrato,
    'Residuo': residui_ang})

fig = px.scatter(
    df_residui, 
    x='Previsto', 
    y='Residuo',
    title='Analisi dei Residui: Errore Inclinazione (Finestra Operativa ±15°)',
    labels={
        'Previsto': 'Errore Previsto dal Modello (°)', 
        'Residuo': 'Residuo (Reale - Previsto) (°)'
    },
    template='plotly_white',
    hover_data=['Reale'])

fig.add_hline(y=0, line_dash="dash", line_color="red", line_width=2)
fig.show()

In [8]:
#ANALISI DEI RESIDUI INCLINAZIONE - ISTOGRAMMA
fig_hist = px.histogram(
    df_residui, 
    x='Residuo',
    nbins=30,
    title='Distribuzione dei Residui: Errore Inclinazione (Finestra ±15°)',
    labels={'Residuo': 'Residuo (Reale - Previsto) (°)'},
    template='plotly_white',
    marginal='box',
    opacity=0.75)

fig_hist.add_vline(x=0, line_dash="dash", line_color="red", line_width=2)
fig_hist.show()

In [9]:
warnings.filterwarnings("ignore", category=ConvergenceWarning)

modello_base_apo = stacking 
modello_base_ang = stacking

modello_vincitore_apogeo = clone(modello_base_apo)   
modello_vincitore_angolo = clone(modello_base_ang)

#ADDESTRAMENTO FINALE
modello_vincitore_apogeo.fit(x_norm, y1)
modello_vincitore_angolo.fit(x_norm, y2)

#SALVATAGGIO
joblib.dump(modello_vincitore_apogeo, "modello_apogeo_v2.pkl")
joblib.dump(modello_vincitore_angolo, "modello_angolo_v2.pkl")
joblib.dump(scaler, "scaler_v2.pkl")

['scaler_v2.pkl']

In [10]:
#CARICAMENTO MODELLI --> RIPARTIRE DA QUI SE I MODELLI SONO GIA' STATI ADDENSTRATI
modello_apo = joblib.load('modello_apogeo_v2.pkl')
modello_ang = joblib.load('modello_angolo_v2.pkl')
scaler_fisso = joblib.load('scaler_v2.pkl')

In [11]:
#FUNZIONE PER CALCOLARE SETUP
def calcola_setup(vento_reale, std_reale, apogeo_obiettivo, modello_apo, modello_ang, scaler):
    lista_zavorre = [0, 20, 40, 60]
    delay_possibili = np.arange(0.8, 3.01, 0.05) 
    
    if vento_reale < 0.2:
        rampe_possibili = [0.0]
    else:
        angolo_min = vento_reale * -5.0
        angolo_max = vento_reale * -3.0
        rampe_possibili = np.arange(angolo_min, angolo_max + 0.1, 0.1)

    combinazioni = list(itertools.product([vento_reale], [std_reale], lista_zavorre, delay_possibili, rampe_possibili))
    df = pd.DataFrame(combinazioni, columns=['vento_medio', 'std_vento', 'zavorra', 'delay_secondi', 'angolo_rampa'])
    
    dati_scalati = scaler.transform(df)
    
    df['Prev_Apogeo'] = modello_apo.predict(dati_scalati)
    df['Prev_Errore_Inclinazione'] = modello_ang.predict(dati_scalati)
    
    ottimali = []
    tolleranza_metri = 5.0
    tolleranza_angolo_sicuro = 1.0 
    
    for z in lista_zavorre:
        df_z = df[df['zavorra'] == z].copy()
        
        df_z_target = df_z[np.abs(df_z['Prev_Apogeo'] - apogeo_obiettivo) <= tolleranza_metri]
        
        if not df_z_target.empty:
            migliore = df_z_target.iloc[np.abs(df_z_target['Prev_Errore_Inclinazione']).argsort()[:1]].iloc[0]
            ottimali.append(migliore)
            
        else:
            df_z_dritti = df_z[np.abs(df_z['Prev_Errore_Inclinazione']) <= tolleranza_angolo_sicuro]
            
            if not df_z_dritti.empty:
                distanza_dal_target = np.abs(df_z_dritti['Prev_Apogeo'] - apogeo_obiettivo)
                migliore_ripiego = df_z_dritti.iloc[distanza_dal_target.argsort()[:1]].iloc[0]
                ottimali.append(migliore_ripiego)

    if not ottimali:
        return "ALLARME ROSSO: Nessun setup sicuro o in target trovato per nessuna zavorra."
        
    df_finale = pd.DataFrame(ottimali)
    df_finale = df_finale.round({'delay_secondi': 2, 'angolo_rampa': 1, 'Prev_Apogeo': 1, 'Prev_Errore_Inclinazione': 1})
    
    return df_finale[['zavorra', 'angolo_rampa', 'delay_secondi', 'Prev_Apogeo', 'Prev_Errore_Inclinazione']]

In [12]:
print("Cucinando setup ottimali...\n")

vento = 4.8
std = 0.1 * vento
apogeo_obiettivo = 200

print(f"Vento: {vento} m/s | Std: {std} m/s")
tabella = calcola_setup(vento, std, apogeo_obiettivo, modello_apo, modello_ang, scaler_fisso)
print(tabella.to_string(index=False))

Cucinando setup ottimali...

Vento: 4.8 m/s | Std: 0.48 m/s
 zavorra  angolo_rampa  delay_secondi  Prev_Apogeo  Prev_Errore_Inclinazione
     0.0         -22.6           2.55        205.0                     -17.3
    20.0         -18.3           2.85        203.8                      -1.0
    40.0         -18.3           2.00        198.3                       0.1
    60.0         -18.4           0.85        200.3                      -0.0
